```yaml
title: "Chapter 1 — Corporate Valuation & Financial Modelling"
authors: "Kate Lawal, Nathan Burns, Isijola Olufemi, Kundan Singh"
affiliation: "Raymond A. Mason School of Business, William & Mary"
business_case: "Automating intrinsic value calculation and investment memo generation from public 10-K data via agentic workflows."
agentic_stack: "Google Gemini 2.5 Flash + smolagents + yfinance + Gradio"
```

---

# Chapter 1 — Corporate Valuation & Financial Modelling
## *Part I: Money Magic*

**The Case:** Every stock has a price, which is what the market thinks a company is worth today. But what is it actually worth, based on the cash it will generate for years to come? Those two numbers are often different. Sometimes wildly so. The gap between them is where every smart investment thesis lives and where every bad one dies. We're building a digital assistant that can evaluate that difference, super fast.

**Tools We Use:** Google Gemini 2.5 Flash (our smart brain) · smolagents (our robot helpers) · yfinance (for company data) · Gradio (to make it a cool app).


---

## §1 — The Detective's Challenge

Imagine you're a detective, but for money. A new company pops up, and you need to quickly figure out if it's a good investment. What do you do?

First, you'd check its money story (financials). Then, you'd run some quick checks (ratios). Next, you'd try to predict its future value (like guessing how much treasure it'll find!). Finally, you'd see what other similar companies are worth. Usually, this takes a lot of time – days, even! But our digital helper can do it in minutes.

Why does this matter? If you get the numbers wrong, someone could lose millions! But even worse than getting it wrong is *missing* something important because you ran out of time. Our helper makes sure we check all the important boxes, every single time. No more skipping important clues!

---

## §2 — Our Super Smart Helper (The Agent)

Think of our helper as a super-organized financial detective. It has a bunch of special gadgets (we call them 'tools'). When you ask it a question, it figures out which tools it needs to use, step-by-step.

For example, to understand a company's money, it might use a 'fetch financials' tool. To guess its future value, it uses a 'DCF model' tool. Each tool does one specific job, and our helper knows exactly when to use each one.

This makes everything super clear. We can see exactly which tool gave us each piece of information. If our helper makes a mistake, it's usually because it picked the wrong tool for the job, not because it did the math wrong. It's like a new kind of puzzle where we train our helper to pick the *right* gadget every time!

---

Before diving into financial analysis, we set up our environment:

### 1. Installing Necessary Libraries

This block handles installing required Python libraries (`smolagents`, `yfinance`, `pandas`, `numpy`, etc.) using `pip`.

To keep our notebook tidy and only show what's important:
* We run the installation quietly, so you don't see all the technical details unless there's a problem.
* If everything goes well, you'll see a green checkmark and a success message.
* If something goes wrong, it will clearly show you the error message from the installation, so we know exactly what to fix.

In [1]:
import subprocess

command = ['pip', 'install', 'smolagents', 'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy', 'gradio', 'pillow']
process = subprocess.run(command, capture_output=True, text=True, check=False)

if process.returncode != 0:
    print(f"Installation failed with errors:\n{process.stderr.strip() or process.stdout.strip()}")
else:
    print("✅ All necessary libraries are installed!")

✅ All necessary libraries are installed!


### 2. Importing all other relevant modules

This block imports essential modules, making their functionalities available:
* `smolagents`: Our core framework for building the intelligent agent.
* `yfinance as yf`: For fetching financial data from Yahoo Finance.
* `pandas as pd`: For data manipulation (e.g., financial statements).
* `numpy as np`: For numerical operations.

A `✅ Imports done` message confirms readiness.

**Alternate for `yfinance`:**
If you prefer other financial data sources, you could use libraries like:
* `import pandas_datareader as pdr`
* `import yahoofinancials as yfs`
* `import requests` (for custom API calls)

Remember to adjust the code that uses `yf.Ticker` accordingly if you switch data providers.

In [2]:
# Standard imports
from smolagents import CodeAgent, tool
from smolagents import OpenAIServerModel
from smolagents.monitoring import LogLevel
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import scipy.stats as stats
from PIL import Image
import io, json, warnings
warnings.filterwarnings('ignore')
print("✅ Imports done")

✅ Imports done


### 3. Connecting our Super Brain (Gemini API Key)

To make our smart helper truly smart, we connect it to a powerful AI brain, the Gemini 2.5 Flash model. This connection uses a special 'API Key'. Think of it as a secret password that allows our notebook to talk to Gemini's brain. You'll usually keep this key safe in Colab's 'Secrets' manager.

In [3]:
# Connect your Gemini API key
# Get a free key at: https://aistudio.google.com/app/apikey
# Then add it to Colab Secrets (key name: GEMINI_API_KEY)

from google.colab import userdata
API_KEY = userdata.get("GEMINI_API_KEY")

model = OpenAIServerModel(
    model_id="gemini-2.5-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=API_KEY,
)
print("✅ Model ready")

✅ Model ready


## §3 — Our Detective's Gadgets (The Tool Catalog)

Every good detective needs a toolkit, and our smart helper is no different! Here are the special tools it uses to figure out a company's financial story and future value. We'll focus on the core horizons: understanding the company's past (Exploratory Data Analysis - EDA) and predicting its future (Discounted Cash Flow- DCF).

| Tool Name | Category | Inputs | Returns | Execution Trigger |
| :--- | :--- | :--- | :--- | :--- |
| `fetch_and_summarise_financials` | EDA | `ticker` (str) | str | First — always. This is reading the case file before asking any questions. |
| `calculate_capm_wacc` | Ratio | `ticker` (str), risk parameters | str | Before the DCF. Sets the discount rate — the lens through which all future money is valued. |
| `run_dcf_model` | Predictive | `ticker` (str), modelling parameters | str | The main investigation. Estimates what the company is actually worth today. |
| `generate_investment_memo` | Action | `ticker` (str), parameters | str | The closing report. Everything gathered, organised, and delivered in one place. |

---

### Gadget 1: The 'Money Scanner' Tool (`fetch_and_summarise_financials`)

**What it does:** This tool is like a special scanner that quickly reads a company's financial report and gives us a super-short summary. It tells us how much money the company brought in (revenue) and how much profit it actually kept (net income). It's always the first tool our helper uses to get a basic understanding.


In [4]:
@tool
def fetch_and_summarise_financials(ticker: str) -> str:
    """
    Fetches and summarises a company's financial statements from Yahoo Finance.

    When to use: Call this FIRST for any valuation. It helps us understand
    the company's money story: how much it earns and how much profit it keeps.

    Args:
        ticker: The stock ticker symbol, e.g. 'MSFT' or 'AAPL'.

    Returns:
        A simple summary of its main money-making and profit numbers.
    """
    co = yf.Ticker(ticker)
    inc = co.income_stmt

    if inc.empty:
        return f"No financial data found for {ticker}."

    # Ensure income statement is transposed to access by column name (date as index)
    inc = inc.transpose()

    revenue_latest = inc.get("Total Revenue", 0)
    net_income_latest = inc.get("Net Income", 0)

    return (
        f"=== {ticker} Quick Financial Snapshot ===\n"
        f"  Latest Annual Revenue: ${revenue_latest/1e9:.1f} Billion\n"
        f"  Latest Annual Net Income: ${net_income_latest/1e9:.2f} Billion\n"
    )

# Quick test to see it in action! (no API key needed)
# print(fetch_and_summarise_financials("MSFT"))

Here, we're simply telling our helper which company to look for by its stock symbol (like 'MSFT' for Microsoft). Then, our helper uses `yf.Ticker` to grab all the public information about that company from Yahoo Finance. Easy peasy!

After finding the income statement, we check if it's empty. If it is, it means we don't have enough data for this company, and our helper will let us know.

Here, we extract the most important numbers: **Total Revenue** (all the money the company made) and **Net Income** (the profit left after all costs). We then put it all into a simple, easy-to-read summary. This gives us a basic 'report card' for the company!

---

### Gadget 2: The 'Cost-of-Money Calculator' Tool (`calculate_capm_wacc`)

**What it does:** This tool figures out the 'Weighted Average Cost of Capital' (WACC). Think of WACC as the company's average cost of getting money to run its business, whether from investors buying shares or from borrowing money. It's super important because it helps us know how much future money is 'worth' today. A dollar you get next year is worth a little less than a dollar today, and WACC helps us figure out *how much* less.

**Why we need it:** We use this 'cost of money' to 'discount' future predictions in our next tool, the DCF model. Without it, we can't accurately guess a company's future value! Get WACC wrong and the entire DCF model shifts. A rate that is two percentage points too low can turn a fairly-valued stock into an apparent screaming buy. This tool makes sure we calculate it properly, every time, from live data


In [5]:
@tool
def calculate_capm_wacc(
    ticker: str,
    risk_free_rate: float = 0.043,
    equity_risk_premium: float = 0.055,
    tax_rate: float = 0.21,
) -> str:
    """
    Calculates the Weighted Average Cost of Capital (WACC).

    WACC is like the 'discount rate' we use in our DCF model. It tells us
    how much a future dollar is worth today, considering the risks.
    A higher WACC means future money is worth less today.

    Formula (Simplified):
        Cost of Equity = Risk-Free Rate + Beta (how shaky the stock is) × Equity Risk Premium
        WACC = (Weight of Equity × Cost of Equity) + (Weight of Debt × Cost of Debt × (1 - Tax Rate))

    When to use: Always calculate this BEFORE running the DCF model, as the DCF needs this number.

    Args:
        ticker: Stock ticker.
        risk_free_rate: The return you can get without risk (like a government bond).This is 5.18% in the US, as at May 2026.
        equity_risk_premium: Extra return investors expect from stocks over safe investments. This is very company specific.
        tax_rate: The company's tax rate. It is currently 21% in the US

    Returns:
        WACC percentage and its main parts, explained simply.
    """
    co   = yf.Ticker(ticker)
    info = co.info # Use .info for more comprehensive data like beta
    bal  = co.balance_sheet.transpose()
    inc  = co.income_stmt.transpose()

    # --- 1. Cost of Equity (CAPM) ---
    # Get beta from info, default to 1.0 if not found
    beta = info.get('beta', 1.0)
    cost_of_equity = risk_free_rate + beta * equity_risk_premium

    # --- 2. Cost of Debt (pre-tax) ---
    # Attempt to estimate cost of debt. If interest expense and total debt are available.
    total_debt = bal.get('Total Debt', bal.get('Long Term Debt', 0))
    interest_expense = inc.get('Interest Expense', 0)
    cost_of_debt = interest_expense / total_debt if total_debt > 0 else 0.05 # Default if no debt/interest

    # --- 3. Market Value of Equity & Debt ---
    market_cap = info.get('marketCap', 0)
    total_debt_value = total_debt # Using book value as a proxy for market value of debt

    # --- 4. Weights of Equity and Debt ---
    total_capital = market_cap + total_debt_value
    weight_of_equity = market_cap / total_capital if total_capital > 0 else 1.0
    weight_of_debt = total_debt_value / total_capital if total_capital > 0 else 0.0

    # --- 5. Calculate WACC ---
    wacc = (weight_of_equity * cost_of_equity) + \
           (weight_of_debt * cost_of_debt * (1 - tax_rate))

    if np.isnan(wacc):
        # Default to a common WACC if calculation fails due to missing data
        wacc = 0.09
        return f"Could not calculate WACC for {ticker} due to missing data. Defaulting to {wacc:.2%}."

    return (
        f"=== {ticker} WACC Calculation ===\n"
        f"  Beta: {beta:.2f}\n"
        f"  Cost of Equity: {cost_of_equity:.2%}\n"
        f"  Cost of Debt (pre-tax): {cost_of_debt:.2%}\n"
        f"  Weight of Equity: {weight_of_equity:.2%}\n"
        f"  Weight of Debt: {weight_of_debt:.2%}\n"
        f"  Tax Rate: {tax_rate:.2%}\n"
        f"  Calculated WACC: {wacc:.2%}"
    )

# Quick test to see it in action!
# print(calculate_capm_wacc("MSFT"))

The most important number for our 'Future Teller' is **Free Cash Flow (FCF)**. This is the actual cash a company has left over after paying for its operations and growth. It's like the real 'treasure' the company generates! If a company has negative FCF, it means it's spending more cash than it's making, which is a red flag for this kind of prediction.

In [6]:
@tool
def run_dcf_model(
    ticker: str,
    wacc: float = 0.09,
    terminal_growth_rate: float = 0.025, #Usually the GDP of the country or overall industry long term growth rate
    forecast_years: int = 5,
) -> str:
    """
    Runs a 5-year Discounted Cash Flow (DCF) model to guess the company's true value.

    The DCF model predicts how much 'free cash' a company will make in the future,
    then 'discounts' that money back to today's value (because money today is worth
    more than money tomorrow!). It then adds a 'terminal value' for all the cash
    it will make even after our 5-year forecast.

    The final result is our best guess of what the stock is truly worth today.

    When to use: This is the main step to find the company's 'true' worth.
    Call this AFTER you've calculated the WACC.

    Args:
        ticker: Stock ticker.
        wacc: The discount rate (from our `calculate_capm_wacc` tool).
        terminal_growth_rate: How much we think the company will grow forever after 5 years.
        forecast_years: How many years into the future we want to predict (usually 5).

    Returns:
        The estimated 'true' value per share, compared to its current price, and if it's
        'cheap' (undervalued) or 'expensive' (overvalued).
    """
    co  = yf.Ticker(ticker)
    cf  = co.cashflow.transpose()
    info = co.info # Use .info for full info
    bal  = co.balance_sheet.transpose()

    # Free Cash Flow (FCF) is the 'treasure' a company has left after paying for everything.
    # It's the most important number for our prediction!
    # Note: Yahoo Finance often presents CAPEX as a negative number
    # "Capital Expenditures" or "Capital Expenditure"
    ocf_key = 'Operating Cash Flow'
    capex_key = 'Capital Expenditures' # or 'Capital Expenditure'

    if ocf_key not in cf.columns:
        return f"{ticker}: No Operating Cash Flow data found. DCF model cannot be run."
    if capex_key not in cf.columns:
        capex_key = 'Capital Expenditure' # Try alternative key
        if capex_key not in cf.columns:
            return f"{ticker}: No Capital Expenditure data found. DCF model cannot be run."

    # Get the latest year's operating cash flow and capex
    ocf = cf.loc[cf.index[0], ocf_key] if not cf.empty and ocf_key in cf.columns else 0
    capex = cf.loc[cf.index[0], capex_key] if not cf.empty and capex_key in cf.columns else 0

    fcf = ocf + capex  # Note: capex is usually shown as a negative number in Yahoo data

    if fcf <= 0:
        return f"{ticker}: Hmm, this company has negative free cash flow. DCF might not be the best tool here. Estimated value: $0.00/share."

    # --- Project FCF for forecast_years ---
    # For simplicity, let's assume FCF grows at a constant rate for the forecast years.
    # In a real model, this would be more detailed.
    projected_fcf = [fcf * (1 + terminal_growth_rate)**i for i in range(1, forecast_years + 1)]

    # --- Discounted FCF ---
    discounted_fcf = [fcf_val / ((1 + wacc)**i) for i, fcf_val in enumerate(projected_fcf, 1)]

    # --- Terminal Value Calculation ---
    # TV = FCF_last_forecast_year * (1 + terminal_growth_rate) / (wacc - terminal_growth_rate)
    # Ensure wacc > terminal_growth_rate to avoid division by zero or negative
    if wacc <= terminal_growth_rate:
        return f"{ticker}: WACC ({wacc:.2%}) must be greater than Terminal Growth Rate ({terminal_growth_rate:.2%}) for DCF model. Estimated value: $0.00/share."

    terminal_fcf = projected_fcf[-1] * (1 + terminal_growth_rate)
    terminal_value = terminal_fcf / (wacc - terminal_growth_rate)

    # --- Discount Terminal Value ---
    discounted_terminal_value = terminal_value / ((1 + wacc)**forecast_years)

    # --- Enterprise Value (EV) ---
    enterprise_value = sum(discounted_fcf) + discounted_terminal_value

    # --- Equity Value ---
    total_cash = bal.get('Cash And Cash Equivalents', bal.get('Cash', 0))
    total_debt = bal.get('Total Debt', bal.get('Long Term Debt', 0))
    # Ensure cash and debt are positive values
    total_cash = max(0, total_cash)
    total_debt = max(0, total_debt)

    equity_value = enterprise_value + total_cash - total_debt

    # --- Intrinsic Value Per Share ---
    shares_outstanding = info.get('sharesOutstanding', info.get('sharesShort', 0)) # sharesOutstanding is more reliable

    if shares_outstanding <= 0:
        return f"{ticker}: Shares outstanding data not available or zero. Cannot calculate per share value. Estimated value: $0.00/share."

    intrinsic = equity_value / shares_outstanding

    # Let's compare our 'true' value to the current market price!
    current_price = info.get('currentPrice', 0) # Use 'currentPrice' from info
    upside = (intrinsic / current_price - 1) * 100 if current_price > 0 else 0

    # Based on our calculations, should we 'Buy', 'Sell', or 'Hold'?
    signal = "FAIRLY VALUED"
    if upside > 15:
        signal = "UNDERVALUED (Potential BUY!)"
    elif upside < -15:
        signal = "OVERVALUED (Potential SELL!)"

    return (
        f"=== {ticker} DCF Valuation Results ===\n"
        f"  Our estimated 'true' value: ${intrinsic:.2f}/share\n"
        f"  Current market price:     ${current_price:.2f}/share\n"
        f"  Looks like:               {signal} ({upside:+.1f}% difference)"
    )

# Quick test for the DCF model
# print(run_dcf_model("MSFT"))

After calculating the discounted value of all future cash flows (including that 'forever' value), we get the **Enterprise Value** - the total value of the company. To find out what it's worth *just for the shareholders* (you, if you own stock!), we adjust this by subtracting debt and adding cash. Finally, we divide this number by the total number of shares to get our estimated **Intrinsic Value Per Share** - the 'true' value of one share according to our model! The previous code cell now contains the complete implementation.

The last step in our 'Future Teller' tool is to compare our estimated 'true' value per share with what the stock is actually selling for today. If our 'true' value is much higher, it might be a 'BUY'! If it's much lower, maybe a 'SELL'. This comparison gives us a simple signal to understand if the stock is 'cheap' or 'expensive' right now.

If our intrinsic value is meaningfully higher than the current price, the stock looks undervalued — a potential buy. If it is meaningfully lower, the stock looks overvalued — a potential sell. Sensitivity analysis is usually done at this point to justify how wrong this intrinsic value is but still be valid with respect to the current price. Remember, no single-point projection is accurate enough.

---

### Gadget 4: The 'Report Card Generator' Tool (`generate_investment_memo`)

**What it does:** This tool brings everything together! It's like a secretary who takes all the information from our 'Money Scanner' and 'Future Teller' tools and writes a simple, easy-to-read report card (we call it an 'Investment Memo') for the company. It gives a quick opinion: Buy, Hold, or Sell.

**Why we need it:** This is the final deliverable! It makes sure all our hard work is presented in a clear, understandable and actionable way, like the summary at the end of a detective's case file.


In [7]:
from smolagents import tool

@tool
def generate_investment_memo(ticker: str, wacc: float = 0.09, tgr: float = 0.025) -> str:
    """
    Generates a simple investment memo, like a report card for the company.

    When to use: Call this as the FINAL step. It brings together our main findings.
    It's like the summary page you'd show your friends!

    Args:
        ticker: Stock ticker.
        wacc: The discount rate used in our value prediction.
        tgr: The long-term growth rate used in our value prediction.

    Returns:
        A short, structured report: Company overview, our predicted value, and a simple
        'Buy / Hold / Sell' idea.
    """
    # Our helper agents will call these key tools to get the info needed for the memo
    financials  = fetch_and_summarise_financials(ticker)
    dcf         = run_dcf_model(ticker, wacc=wacc, terminal_growth_rate=tgr)

    # We'll figure out the 'Buy' or 'Sell' recommendation based on our DCF model.
    signal = "HOLD (Wait and see)" # Default recommendation
    if "UNDERVALUED" in dcf:
        signal = "BUY (Looks like a good deal!)"
    elif "OVERVALUED" in dcf:
        signal = "SELL (Might be too expensive!)"

    memo = f"""
╔══════════════════════════════════════════════════════╗
║           INVESTMENT REPORT — {ticker:<6}                ║
╚══════════════════════════════════════════════════════╝

**Our Quick Opinion:** {signal}
(Based on our prediction of its true value compared to today's price)

-- COMPANY'S MONEY STORY (Financials) ------------------
{financials}

-- PREDICTING TRUE VALUE (DCF Model) -------------------
{dcf}

-- IMPORTANT NOTE --------------------------------------
This report was made by our smart AI helper using public information.
It's just for learning and fun, NOT real financial advice!
"""
    return memo.strip()

# Let's get a full report for Apple!
# print(generate_investment_memo("AAPL"))

Based on the DCF model's results (whether the stock is 'undervalued' or 'overvalued'), this tool makes a simple recommendation: BUY, SELL, or HOLD. Then, it neatly organizes all the information into a final, easy-to-read Investment Report, complete with a clear recommendation and an important disclaimer that it's for learning, not actual financial advice!

This tool acts as the final reporter. It doesn't do new calculations but calls our previous tools (`fetch_and_summarise_financials` and `run_dcf_model`) to get their results. This way, all the important findings are collected in one place.

## §4 — Orchestration

A detective does not just carry tools — they know which one to pick up next. That coordination is what `CodeAgent` from smolagents handles. Rather than running a fixed script, the agent reads each tool's instructions (the docstrings) and decides what to call based on what it already knows.

For a standard valuation investigation, the case unfolds in this order:

```
Company ticker arrives
        |
        v
fetch_and_summarise_financials  ← read the case file first
        |
        v
calculate_capm_wacc             ← establish how much future money is worth
        |
        v
run_dcf_model                   ← follow the cash to its conclusion
        |
        v
generate_investment_memo        ← write the closing report
        |
        v
Gradio interface                ← hand it to the client
```

The agent can adapt if needed — if upstream data is already available, it can skip ahead. That flexibility is what makes an agent different from a simple pipeline. The code below sets it all in motion:

In [8]:
# Initialize the Gemini model agent interface via smolagents protocols
API_KEY = userdata.get("GEMINI_API_KEY")
if API_KEY:
    agent_model = OpenAIServerModel(
        model_id="gemini-2.5-flash",
        api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=API_KEY,
    )

    valuation_agent = CodeAgent(
        tools=[fetch_and_summarise_financials, calculate_capm_wacc, run_dcf_model, generate_investment_memo],
        model=agent_model,
        verbosity_level=LogLevel.INFO
    )
    print("✅ Intelligent Financial Agent initialized with multi-tool capabilities.")
else:
    print("⚠️ Set GEMINI_API_KEY in secrets to activate model orchestration processing cells.")

✅ Intelligent Financial Agent initialized with multi-tool capabilities.


## §5 — Interface

The investigation is complete. Now we open the front desk.

The Gradio interface turns the entire agent into a one-click web app. A user types a ticker, presses the button, and receives a full investment memo — no Python required, no spreadsheet to maintain, no case file to assemble manually. A portfolio manager, an MBA student, an analyst who wants a fast second opinion — anyone can walk up and ask the question.

In [9]:
import gradio as gr

def run_agent_analysis(ticker_symbol):
    if not API_KEY:
        return "Please configure API tracking key credentials in your notebook environment settings first."
    try:
        response = valuation_agent.run(f"Generate a full investment report package and analysis for ticker asset {ticker_symbol.strip().upper()}")
        return response
    except Exception as error:
        return f"Operational exception encountered: {str(error)}"

app_interface = gr.Interface(
    fn=run_agent_analysis,
    inputs=gr.Textbox(label="Target Asset Ticker (e.g. TXT, LMT, CMG)", value="TXT"),
    outputs=gr.Markdown(label="Executive Output Dossier"),
    title="🏦Corporate Valuation Dashboard",
    description="Automate intrinsic asset evaluation via real-time balance sheet modeling blocks."
)

# To open the live framework, uncomment and run the line below:
# app_interface.launch(quiet=True)

## §6 — Discussion: Traditional vs. Agentic

Here is an honest accounting of what changed — and what did not.

| Task | What the analyst used to do | What the agent does | The difference |
| :--- | :--- | :--- | :--- |
| **Pull financial data** | Log into SEC EDGAR or Bloomberg, download, format | `fetch_and_summarise_financials` via yfinance | Hours → seconds |
| **Calculate WACC** | Build a dedicated Excel tab, look up beta manually | `calculate_capm_wacc` from live data | 30 minutes → instant |
| **Run the DCF model** | 40-row Excel model, circular references, manual stress-testing | `run_dcf_model` with transparent Python | 2–4 hours → seconds |
| **Write the memo** | Word document, cross-referencing three spreadsheets | `generate_investment_memo` | 30–60 minutes → seconds |

The underlying analysis has not changed. The same revenue figures, the same WACC formula, the same Gordon Growth terminal value. The agent did not invent new finance — it made existing finance faster to run and harder to skip.

### Three things that can still go wrong

**Confabulation risk:** the agent does not invent financial numbers — every figure comes from Yahoo Finance or a formula. The real risk is in the narrative. An agent that has the right numbers can still draw the wrong conclusion in the memo text. Read the evidence, not just the verdict.

**Sequence risk:** an agent that generates a BUY recommendation without first running `calculate_capm_wacc` has skipped a critical step — and the output looks confident either way. Always review the agent's tool-call trace, not just its final answer. The trail matters.

**Assumption risk:** the DCF model assumes a growth rate for five years and a terminal rate forever. Move either number by two percentage points and the intrinsic value shifts substantially. The model's output is only as good as the assumptions going in. That has always been true, and our digital detective is no exception.

## §7 — Reflection (Shared Prompts)

Every good investigation ends with a debrief. What worked? What did not? What would we do differently next time?

What broke first? The evidence retrieval. Some companies report capital expenditures as 'Capital Expenditures' (plural); others use 'Capital Expenditure' (singular). The first version of the DCF tool crashed silently on companies using the singular form — it looked for the wrong column name and found nothing. The fix: always check for both variations before trying to read the data. Even one character of difference can close a case prematurely.

What surprised us? The agent's composure under pressure. When interest expense data was missing — common for companies with no debt — it did not panic or crash. It defaulted to a standard industry proxy rate and noted it clearly in the output. That kind of transparent fallback is far easier to catch and correct than a spreadsheet that silently leaves a cell blank.

Where did the agent overstep? Early in testing, the model began writing its own valuation narrative rather than running the structured tools — confident-sounding prose with no calculation behind it. The fix was not a stricter prompt; it was better tool docstrings. Specifically, the When to use line in each tool tells the agent when it must reach for that instrument rather than improvising. Write better case instructions, get better detective work.


## §8 — Try It Yourself

1.  **Dataset Swap Exercise:** Run the agent on `CMG` (Chipotle) and `LMT` (Lockheed Martin). One is a high-growth consumer brand; the other is a stable defence contractor. Feed both the same DCF assumptions and see how differently the intrinsic values land — then ask yourself whether identical assumptions even make sense across two such different businesses.

2. **Adjust the lens.** In `calculate_capm_wacc`, raise `tax_rate` from `0.21` to `0.28`. Watch how a higher tax burden changes the WACC, and how that change flows through to the final intrinsic value. This is how policy changes show up in valuations.

3. **Add a new instrument to the kit.** Write a function called `calculate_altman_zscore` that computes the five-factor bankruptcy risk score. Register it in the `CodeAgent` tool list. Then update `generate_investment_memo` to include the result at the top — before the Buy / Hold / Sell verdict. A detective who recommends a buy without checking bankruptcy risk has missed an obvious lead.

---

##Other Tools (for the curious minds!)

In a complete financial toolkit, there are many other gadgets! For this simple introduction, we focused on the most important ones for Exploratory Data Analysis (like looking at basic financials) and Discounted Cash Flow (predicting future value).

However, our smart helper could also use tools for:

* **Profitability Ratios:** To see how good a company is at turning sales into profit.
* **Liquidity & Leverage:** To check if a company has enough cash to pay its bills and isn't borrowing too much.
* **Bankruptcy Risk (Altman Z-Score):** To quickly see if a company is in danger of going out of business.
* **Sensitivity Analysis:** To understand how our predictions might change if our guesses about growth rates or costs of money are a little off.

These tools are super helpful, but we've kept them aside for now to keep things simple and focused on the core ideas of EDA and DCF!

In [ ]:
# --- MASTER GOOGLE DRIVE TO GITHUB ROOT PUSH ---
import os
from google.colab import drive

try:
    # 1. Mount Google Drive so the environment can see your folders
    print("🔄 Connecting to your Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # 2. Retrieve secure credentials from Colab Secrets
    GIT_TOKEN    = userdata.get('GIT_TOKEN_ebook')
    GIT_USERNAME = userdata.get('GITHUB_USER')
    GIT_EMAIL    = userdata.get('GITHUB_EMAIL')
    GIT_REPO     = userdata.get('GITHUB_REPO_2')
    NOTEBOOK     = "01_corporate_valuation.ipynb"

    if not GIT_TOKEN or not GIT_USERNAME or not GIT_REPO:
        raise ValueError("Missing parameters! Make sure GITHUB_USER, GITHUB_REPO_2, and GIT_TOKEN_ebook are active in your secrets.")

    # 3. Build secure destination route path
    GIT_PATH = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"

    # 4. Configure Git global identity profiles
    !git config --global user.email "{GIT_EMAIL}"
    !git config --global user.name "{GIT_USERNAME}"

    # 5. Clean out old temporary sync folders and clone a fresh tracking workspace
    %cd /content
    if os.path.exists(f"/content/{GIT_REPO}"):
        !rm -rf {GIT_REPO}
    !git clone {GIT_PATH}

    # 6. Establish exact copy pathways from your active Google Drive directory
    # Points to: My Drive -> Colab Notebooks -> AI_agents_across_businesses
    DRIVE_SRC_PATH = f"/content/drive/MyDrive/Colab_Notebooks/AI_agents_across_businesses/{NOTEBOOK}"
    REPO_DEST_PATH = f"/content/{GIT_REPO}/{NOTEBOOK}"

    if not os.path.exists(DRIVE_SRC_PATH):
        raise FileNotFoundError(f"Could not locate the notebook in your Drive path: {DRIVE_SRC_PATH}\nDouble-check your folder capitalization in Google Drive!")

    print(f"🎯 Target file located in Google Drive! Splicing tracking layer...")
    !cp "{DRIVE_SRC_PATH}" "{REPO_DEST_PATH}"

    # 7. Navigate into the repo and push straight to the root level of the main branch
    %cd /content/{GIT_REPO}

    timestamp = pd.to_datetime('now').strftime('%Y-%m-%d %H:%M')
    !git add {NOTEBOOK}
    !git commit -m "Chapter 1 Master Deploy Sync - {timestamp}" --allow-empty
    !git branch -M main

    print("\n🚀 Executing clean forced push directly to your main repository root...")
    !git push -u origin main --force

    print(f"\n✅ SUCCESS! Your master file is live at the root layer of your repository.")
    print(f"🔗 Refresh your browser window here: https://github.com/{GIT_USERNAME}/{GIT_REPO}")

except Exception as error:
    print(f"\n❌ Synchronization pipeline failed: {error}")

🔄 Connecting to your Google Drive...
